# Background

This notebook simply replicates the experiments in weakest_link_classification.ipynb and uploads them to mlflow.

## Load input datasets

In [1]:
from typing import List

from pathlib import Path
import numpy as np
import pandas as pd

# Models
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

# Boosting models
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

# Preprocessing & evaluation
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_validate
)
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)
from sklearn.base import clone

# Environment configuration
from dotenv import load_dotenv
from os import getenv

# mlflow
import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature

# Warning ignores
import warnings
from urllib3.exceptions import InsecureRequestWarning
warnings.filterwarnings("ignore", category=InsecureRequestWarning)

import matplotlib.pyplot as plt

In [11]:
# === Load input datasets ===

DATA_DIR = Path("../../data/processed")
scores_path = DATA_DIR / "20190108_scores_and_weak_links.csv"
big_path = DATA_DIR / "AimoScore_WeakLink_big_scores.csv"

scores = pd.read_csv(scores_path)
big = pd.read_csv(big_path)

print(scores.shape, big.shape)
print(scores.columns[:10])
print(big.columns[:10])


(2096, 17) (2094, 43)
Index(['ID', 'Date', 'SCORE', 'ForwardHead', 'LeftArmFallForward',
       'RightArmFallForward', 'LeftShoulderElevation',
       'RightShoulderElevation', 'ExcessiveForwardLean',
       'LeftAsymmetricalWeightShift'],
      dtype='object')
Index(['AimoScore', 'No_1_Angle_Deviation', 'No_2_Angle_Deviation',
       'No_3_Angle_Deviation', 'No_4_Angle_Deviation', 'No_5_Angle_Deviation',
       'No_6_Angle_Deviation', 'No_7_Angle_Deviation', 'No_8_Angle_Deviation',
       'No_9_Angle_Deviation'],
      dtype='object')


## Create WeakestLink labels (rule-based)

In [12]:
# === Create WeakestLink labels ===

NON_INDICATORS = {"ID", "Date", "SCORE"}
indicator_cols = [c for c in scores.columns if c not in NON_INDICATORS]

DROP_WEAK_LINKS = {
    "LeftHeelRises",
    "RightHeelRises",
    "LeftKneeMovesInward",
    "RightAsymmetricalWeightShift"
}

indicator_cols = [c for c in indicator_cols if c not in DROP_WEAK_LINKS]


scores[indicator_cols] = scores[indicator_cols].apply(
    pd.to_numeric, errors="coerce"
)

row_max = scores[indicator_cols].max(axis=1)
ties = (scores[indicator_cols].eq(row_max, axis=0).sum(axis=1) > 1)

scores["WeakestLink"] = scores[indicator_cols].idxmax(axis=1)
scores["HasTie"] = ties

print("Tie rows:", int(scores["HasTie"].sum()))
print("Label null rows:", int(scores["WeakestLink"].isna().sum()))


Tie rows: 372
Label null rows: 0


## Merge labels with feature dataset

In [13]:
# === Merge labels with features ===

labels = scores[["ID", "WeakestLink", "HasTie"]].dropna()
data = big.merge(labels, on="ID", how="inner")

print("After merge:", data.shape)
print("Unique classes:", data["WeakestLink"].nunique())


After merge: (2094, 45)
Unique classes: 10


## Feature selection & cleaning

In [14]:
# === Feature selection & cleaning ===

angle_cols = [f"No_{i}_Angle_Deviation" for i in range(1, 14)]
nasm_cols = [f"No_{i}_NASM_Deviation" for i in range(1, 26)]
feature_cols = angle_cols + nasm_cols

X = data[feature_cols].apply(pd.to_numeric, errors="coerce")
y = data["WeakestLink"].astype(str)

mask = X.notna().all(axis=1)
X, y = X[mask], y[mask]

print("After dropping NaNs:", X.shape, y.shape)


After dropping NaNs: (2094, 38) (2094,)


## Handle ties & singleton classes

In [15]:
# === Handle ties & singleton classes ===

has_tie = data.loc[X.index, "HasTie"].astype(bool)

X_keep, y_keep = X.copy(), y.copy()
X_notie, y_notie = X.loc[~has_tie], y.loc[~has_tie]

counts = y_keep.value_counts()
singletons = counts[counts < 2].index

X2 = X_keep[~y_keep.isin(singletons)]
y2 = y_keep[~y_keep.isin(singletons)]

print("Final dataset:", X2.shape)
print("Smallest class count:", y2.value_counts().min())


Final dataset: (2094, 38)
Smallest class count: 46


## Train/test split & cross-validation setup

In [7]:
# === Train/test split & cross-validation ===

X_train, X_test, y_train, y_test = train_test_split(
    X2, y2,
    test_size=0.2,
    random_state=42,
    stratify=y2
)

min_class = y_train.value_counts().min()
cv_splits = max(2, min(5, min_class))

cv = StratifiedKFold(
    n_splits=cv_splits,
    shuffle=True,
    random_state=42
)

print("Using cv_splits =", cv_splits)


Using cv_splits = 5


## Label encoding (boosting models only)

In [8]:
# === Label encoding (boosting models only) ===

label_encoder = LabelEncoder()
y2_encoded = label_encoder.fit_transform(y2)

class_names = list(label_encoder.classes_)

X_train_enc, X_test_enc, y_train_enc, y_test_enc = train_test_split(
    X2, y2_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y2_encoded
)


## Model definitions

In [12]:
# === Model definitions ===
models = {
    "dummy_most_frequent": DummyClassifier(strategy="most_frequent"),

    "logreg_C1_balanced": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(
            solver="lbfgs",
            max_iter=5000,
            class_weight="balanced",
            C=1.0
        ))
    ]),
    "logreg_C0.3_balanced": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(
            solver="lbfgs",
            max_iter=5000,
            class_weight="balanced",
            C=0.3
        ))
    ]),

    "gaussian_nb": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", GaussianNB())
    ]),

    "lda": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LinearDiscriminantAnalysis())
    ]),

    # stable shrinkage LDA variant
    "lda_lsqr_shrinkage_auto": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto"))
    ]),

    "knn_k5": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", KNeighborsClassifier(n_neighbors=5))
    ]),
    "knn_k15": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", KNeighborsClassifier(n_neighbors=15))
    ]),
    "xgboost": Pipeline([
        ("clf", XGBClassifier(
            n_estimators=300,
            max_depth=6,
            learning_rate=0.1,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="multi:softprob",
            eval_metric="mlogloss",
            num_class=len(class_names),
            random_state=42,
            n_jobs=-1
        ))
    ]),
    "lightgbm": Pipeline([
        ("clf", LGBMClassifier(
            n_estimators=300,
            learning_rate=0.05,
            num_leaves=31,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        ))
    ]),
    "catboost": Pipeline([
    ("clf", CatBoostClassifier(
        iterations=300,
        learning_rate=0.1,
        depth=6,
        loss_function="MultiClass",
        auto_class_weights="Balanced",
        verbose=False,
        random_seed=42,
        train_dir="catboost_tmp"
    ))
]),
}


# Cross-validation comparison and model training

In [13]:
# Mlflow setup

# Loading credentials
env_path = "../../ml.env"
load_dotenv(env_path)

mlflow.set_tracking_uri(getenv("MLFLOW_TRACKING_URI"))


In [14]:
mlflow.set_experiment("A3_weakest_link_comparison")
def flatten_params(params: dict) -> dict:
    flat = {}
    for k, v in params.items():
        if isinstance(v, (str, int, float, bool)) or v is None:
            flat[k] = v
        else:
            flat[k] = str(v)
    return flat



In [15]:
# ============================================================
# === Cross-validation comparison (with run metadata) ===
# Adds columns to results_df so you can see (and screenshot):
# - dataset tag (optional)
# - number of folds (cv_splits)
# - which WeakestLink classes were removed (or None)
# ============================================================

# Optional label to identify this run in screenshots
DATASET_TAG = globals().get("DATASET_TAG", "dataset variant not specified")

# WeakestLink removal info (handles "no dropped weakest link" cleanly)
if "DROP_WEAK_LINKS" in globals() and DROP_WEAK_LINKS:
    removed_links_sorted = sorted(list(DROP_WEAK_LINKS))
    removed_links_str = ", ".join(removed_links_sorted)
else:
    removed_links_str = "None"

# Pull fold count from your earlier logic (or fallback to cv.n_splits if present)
cv_folds = globals().get("cv_splits", getattr(cv, "n_splits", "unknown"))

scoring = {
    "accuracy": "accuracy",
    "precision_macro": "precision_macro",
    "recall_macro": "recall_macro",
    "f1_macro": "f1_macro",
    "f1_weighted": "f1_weighted",
}

rows = []

with mlflow.start_run(run_name="cv_model_comparison") as parent_run:
    mlflow.log_param("dataset_tag", DATASET_TAG)
    mlflow.log_param("cv_folds", cv_folds)
    mlflow.log_param("removed_weakest_links", removed_links_str)
    mlflow.log_param("num_models_compared", len(models))

    mlflow.set_tags({
        "stage": "comparison",
    })

    # --------------------------------------------------------
    # One child run per model
    # --------------------------------------------------------
    for name, model in models.items():
        with mlflow.start_run(run_name=name, nested=True):
            # Use encoded labels only for xgboost, same as your notebook
            if name == "xgboost":
                Xtr, ytr = X_train_enc, y_train_enc
                label_mode = "encoded"
            else:
                Xtr, ytr = X_train, y_train
                label_mode = "original"

            # 1) Log parameters
            mlflow.log_params(flatten_params(model.get_params(deep=True)))
            mlflow.log_param("model_name", name)
            mlflow.log_param("dataset_tag", DATASET_TAG)
            mlflow.log_param("cv_folds", cv_folds)
            mlflow.log_param("removed_weakest_links", removed_links_str)
            mlflow.set_tags({"label_mode": label_mode})

            # 2) Run CV and log metrics
            out = cross_validate(
                model,
                Xtr,
                ytr,
                cv=cv,
                scoring=scoring,
                n_jobs=-1,
                return_train_score=False
            )

            row = {
                "dataset_tag": DATASET_TAG,
                "cv_folds": cv_folds,
                "removed_weakest_links": removed_links_str,
                "model": name,
            }

            for metric_name in scoring:
                fold_values = out[f"test_{metric_name}"]
                mean_val = float(np.mean(fold_values))
                std_val = float(np.std(fold_values, ddof=1)) if len(fold_values) > 1 else 0.0

                row[f"{metric_name}_mean"] = mean_val
                row[f"{metric_name}_std"] = std_val

                mlflow.log_metric(f"cv_{metric_name}_mean", mean_val)
                mlflow.log_metric(f"cv_{metric_name}_std", std_val)

                for i, fold_val in enumerate(fold_values, start=1):
                    mlflow.log_metric(f"fold_{i}_{metric_name}", float(fold_val))

            rows.append(row)

            # 3) Fit on full training set and log model artifact
            fitted_model = clone(model)
            fitted_model.fit(Xtr, ytr)

            sample_X = Xtr.iloc[:5] if hasattr(Xtr, "iloc") else Xtr[:5]
            sample_pred = fitted_model.predict(sample_X)
            signature = infer_signature(sample_X, sample_pred)

            mlflow.sklearn.log_model(
                sk_model=fitted_model,
                name="model",
                signature=signature,
                input_example=sample_X,
            )

/Users/pettergustafsson/study/4DT907/ml_project/machine_learning_project_4dt907/ml/models/venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/pettergustafsson/study/4DT907/ml_project/machine_learning_project_4dt907/ml/models/venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/pettergustafsson/study/4DT907/ml_project/machine_learning_project_4dt907/ml/models/venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: P

🏃 View run dummy_most_frequent at: https://172.27.62.170:5001/#/experiments/18/runs/ba521a6940bd403e9dd66ad1130ee16c
🧪 View experiment at: https://172.27.62.170:5001/#/experiments/18


2026/03/13 07:00:10 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
/Users/pettergustafsson/study/4DT907/ml_project/machine_learning_project_4dt907/ml/models/venv/lib/python3.11/site-packages/mlflow/models/model.py:1209: FutureWarning: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is the 'skops' format.
  flavor.save_model(path=local_path, mlflow_model=mlflow_model, **kwargs)


🏃 View run logreg_C1_balanced at: https://172.27.62.170:5001/#/experiments/18/runs/fa599b8526e843c98a531d2436b90d34
🧪 View experiment at: https://172.27.62.170:5001/#/experiments/18


2026/03/13 07:00:21 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
/Users/pettergustafsson/study/4DT907/ml_project/machine_learning_project_4dt907/ml/models/venv/lib/python3.11/site-packages/mlflow/models/model.py:1209: FutureWarning: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is the 'skops' format.
  flavor.save_model(path=local_path, mlflow_model=mlflow_model, **kwargs)


🏃 View run logreg_C0.3_balanced at: https://172.27.62.170:5001/#/experiments/18/runs/e3a07864cc40414f89b42b4536d8b8dd
🧪 View experiment at: https://172.27.62.170:5001/#/experiments/18


... (12 outputs left out)

2026/03/13 07:02:17 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
/Users/pettergustafsson/study/4DT907/ml_project/machine_learning_project_4dt907/ml/models/venv/lib/python3.11/site-packages/mlflow/models/model.py:1209: FutureWarning: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is the 'skops' format.
  flavor.save_model(path=local_path, mlflow_model=mlflow_model, **kwargs)


🏃 View run lightgbm at: https://172.27.62.170:5001/#/experiments/18/runs/b236cd8a1ff444fa913754cfafc73aa3
🧪 View experiment at: https://172.27.62.170:5001/#/experiments/18


/Users/pettergustafsson/study/4DT907/ml_project/machine_learning_project_4dt907/ml/models/venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
2026/03/13 07:02:54 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
/Users/pettergustafsson/study/4DT907/ml_project/machine_learning_project_4dt907/ml/models/venv/lib/python3.11/site-packages/mlflow/models/model.py:1209: FutureWarning: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is the 'skops' format.
  flavor.save_model(path=local_path, mlflow_mo

🏃 View run catboost at: https://172.27.62.170:5001/#/experiments/18/runs/1e94969901764b6bbac39ed5e05e0e01
🧪 View experiment at: https://172.27.62.170:5001/#/experiments/18
🏃 View run cv_model_comparison at: https://172.27.62.170:5001/#/experiments/18/runs/3a3421be102242ef9e808a2398d612bd
🧪 View experiment at: https://172.27.62.170:5001/#/experiments/18


In [7]:
# Model loading
from mlflow.models import Model

logistic_regression_version = "5"
knn5_version = "6"
lightgbm_version = "8"
gaussian_nb_version = "10"
lda_version = "11"

logistic_regression = mlflow.sklearn.load_model(f"models:/weakest_link_model/{logistic_regression_version}")
knn5 = mlflow.sklearn.load_model(f"models:/weakest_link_model/{knn5_version}")
lightgbm = mlflow.sklearn.load_model(f"models:/weakest_link_model/{lightgbm_version}")
gaussian_nb = mlflow.sklearn.load_model(f"models:/weakest_link_model/{gaussian_nb_version}")
lda = mlflow.sklearn.load_model(f"models:/weakest_link_model/{lda_version}")



In [8]:
# Prepare input
def process_input(input_features: List[str], values: List[float]) -> pd.DataFrame:
    return pd.DataFrame([values], columns=input_features)

model_uri = f"models:/weakest_link_model/{logistic_regression_version}" # All models in previous cell use the same schema

example = mlflow.artifacts.load_dict(f"{model_uri}/input_example.json")

input_features = example["columns"]
values = example["data"][0]
input = process_input(input_features, values)
input


,No_1_Angle_Deviation,No_2_Angle_Deviation,No_3_Angle_Deviation,No_4_Angle_Deviation,No_5_Angle_Deviation,No_6_Angle_Deviation,No_7_Angle_Deviation,No_8_Angle_Deviation,No_9_Angle_Deviation,No_10_Angle_Deviation,...,No_16_NASM_Deviation,No_17_NASM_Deviation,No_18_NASM_Deviation,No_19_NASM_Deviation,No_20_NASM_Deviation,No_21_NASM_Deviation,No_22_NASM_Deviation,No_23_NASM_Deviation,No_24_NASM_Deviation,No_25_NASM_Deviation
0,0.663797,0.255858,0.863223,0.790053,0.164515,0.329986,0.267336,0.52176,0.699665,0.601148,...,0.34242,0.495457,0.764228,0.670971,0.656624,0.642276,0.825921,0.868962,0.651841,0.814921


In [32]:
# Test prediction
print(logistic_regression.predict(input))
print(knn5.predict(input))
print(lightgbm.predict(input))
print(gaussian_nb.predict(input))
print(lda.predict(input))

['LeftKneeMovesOutward']
['RightArmFallForward']
['RightArmFallForward']
['LeftKneeMovesOutward']
['RightArmFallForward']
